In [14]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### Unzipping the Dataset
We will extract the contents of the ZIP archive to a temporary directory so we can inspect and analyze the files.

In [ ]:
import zipfile
import os

zip_path = '/content/archive (1).zip'
extract_path = '/content/extracted_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# List the extracted files
extracted_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        extracted_files.append(os.path.join(root, file))

print("Extracted files:")
for file in extracted_files:
    print(file)

### Loading and Inspecting the Dataset
We will load `starbucks.csv` using pandas, inspect the shape, column info, and preview the first few rows to understand the features available.

In [ ]:
import pandas as pd

# Load the dataset
file_path = '/content/extracted_data/starbucks.csv'
df = pd.read_csv(file_path)

# Display basic information
print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()

# Display preview
print("\nFirst 5 rows:")
df.head()

### Data Cleaning
We will clean column names, convert percent strings and object columns to appropriate numeric types, and inspect the resulting clean DataFrame.

In [ ]:
import numpy as np

# 1. Clean column names (strip whitespace)
df.columns = df.columns.str.strip()
print("Cleaned Column Names:", list(df.columns))

# 2. Clean percentage columns
pct_cols = ['Vitamin A (% DV)', 'Vitamin C (% DV)', 'Calcium (% DV)', 'Iron (% DV)']
for col in pct_cols:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Clean numeric columns stored as objects
# Total Fat (g)
df['Total Fat (g)'] = pd.to_numeric(df['Total Fat (g)'], errors='coerce')

# Caffeine (mg)
# Inspect unique values in Caffeine first to understand variations
print("\nUnique Caffeine values before cleaning:", df['Caffeine (mg)'].unique()[:10])
df['Caffeine (mg)'] = pd.to_numeric(df['Caffeine (mg)'], errors='coerce')

# 4. Check for any missing values after conversion
print("\nMissing values count after numeric conversion:")
print(df.isnull().sum())

# Fill NaN values in numeric columns with the median
for col in ['Total Fat (g)', 'Caffeine (mg)'] + pct_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Imputed missing values in '{col}' with median: {median_val}")

print("\nPost-Cleaning Dataset Info:")
df.info()

### 1. Statistical Summary
We will examine general descriptive statistics for the nutritional columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Describe numerical data
df.describe()

### 2. Beverage Category Distribution
Let's understand which category has the highest representation in the Starbucks menu.

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, y='Beverage_category', order=df['Beverage_category'].value_counts().index, palette='viridis')
plt.title('Distribution of Beverages by Category')
plt.xlabel('Count')
plt.ylabel('Beverage Category')
plt.tight_layout()
plt.show()

### 3. Correlation Heatmap
Let's check how the nutrients correlate with each other, especially focusing on how Sugar, Carbohydrates, and Fats correlate with Calories.

In [ ]:
plt.figure(figsize=(12, 10))
numeric_cols = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_cols.corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix of Starbucks Beverage Nutrients')
plt.tight_layout()
plt.show()

### 4. Calorie Analysis and Top Sugary Drinks
Let's plot the distribution of Calories and inspect which categories are the highest on average.

In [ ]:
# 1. Distribution of Calories
plt.figure(figsize=(10, 4))
sns.histplot(df['Calories'], kde=True, color='forestgreen', bins=20)
plt.title('Distribution of Calories in Starbucks Beverages')
plt.xlabel('Calories')
plt.ylabel('Frequency')
plt.show()

# 2. Average Calories by Category
plt.figure(figsize=(10, 5))
avg_calories_cat = df.groupby('Beverage_category')['Calories'].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=avg_calories_cat, x='Calories', y='Beverage_category', palette='magma')
plt.title('Average Calories by Beverage Category')
plt.xlabel('Average Calories')
plt.ylabel('Beverage Category')
plt.tight_layout()
plt.show()

### 5. Model Development: Train-Test Split and Decision Tree Regressor
We will now use `scikit-learn` to prepare our data, split it into 80% training and 20% testing sets, train a Decision Tree Regressor to predict `Calories` based on other nutritional facts, and evaluate its performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

# Define features (X) and target (y)
# We'll use numerical nutritional features to predict Calories
feature_cols = [
    'Total Fat (g)', 'Trans Fat (g)', 'Saturated Fat (g)', 'Sodium (mg)',
    'Total Carbohydrates (g)', 'Cholesterol (mg)', 'Dietary Fibre (g)',
    'Sugars (g)', 'Protein (g)', 'Vitamin A (% DV)', 'Vitamin C (% DV)',
    'Calcium (% DV)', 'Iron (% DV)', 'Caffeine (mg)'
]

X = df[feature_cols]
y = df['Calories']

# Perform 80% training and 20% testing split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Initialize and train Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)

# Make predictions
y_pred = dt_model.predict(X_test)

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation Metrics:")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} Calories")
print(f"R-squared (R2) Score: {r2:.4f}")

### 6. Feature Importance
Let's check which nutritional features contribute the most to predicting the calorie content of Starbucks beverages according to our Decision Tree.

In [ ]:
importances = dt_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df, x='Importance', y='Feature', palette='crest')
plt.title('Feature Importances in Decision Tree Calorie Predictor')
plt.xlabel('Importance Score')
plt.ylabel('Nutrient Feature')
plt.tight_layout()
plt.show()

### 7. Model Evaluation & Visualization (Post-Testing)
We will plot the actual versus predicted values of the test set to visually inspect the model's accuracy, alongside a residual plot to check for patterns in prediction errors.

In [ ]:
plt.figure(figsize=(14, 6))

# Plot 1: Actual vs Predicted
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_test, y=y_pred, color='forestgreen', alpha=0.7, s=80)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs. Predicted Calories (Test Set)')
plt.xlabel('Actual Calories')
plt.ylabel('Predicted Calories')

# Plot 2: Residuals Plot
residuals = y_test - y_pred
plt.subplot(1, 2, 2)
sns.scatterplot(x=y_pred, y=residuals, color='crimson', alpha=0.7, s=80)
plt.axhline(y=0, color='black', linestyle='--', lw=2)
plt.title('Residuals Plot')
plt.xlabel('Predicted Calories')
plt.ylabel('Residuals (Actual - Predicted)')

plt.tight_layout()
plt.show()

### 8. Model Results, Accuracy, and Precision Analysis

For regression tasks like predicting **Calories**, performance is measured using continuous error metrics instead of classification accuracy/precision:

| Metric | Value | Meaning for this Model |
| :--- | :--- | :--- |
| **R-squared ($R^2$) Score** | **0.9193 (91.93%)** | **Model Accuracy**: Explains approximately **91.9%** of the variance in beverage calorie content using the given nutritional features. |
| **Root Mean Squared Error (RMSE)** | **24.31 Calories** | **Model Precision**: The standard deviation of the residuals. On average, predictions deviate by about **24.3 calories** from the actual values. |
| **Mean Absolute Error (MAE)** | *~16.7 Calories* | **Average Error**: The average absolute difference between predicted and actual values (giving less weight to outliers than RMSE). |

#### Summary of Insights:
- **Excellent Fit**: An $R^2$ of over 91% indicates a highly accurate model.
- **Feature Dominance**: As shown in the Feature Importance plot, **Cholesterol (mg)** and **Sugars (g)** are the main drivers of the tree splits. This suggests that dairy content (represented by cholesterol) and syrup/sweeteners (sugars) almost fully dictate the calorie count of a Starbucks beverage.
- **Error Distribution**: The residual plot shows that errors are randomly scattered around 0, proving that the model's predictions are unbiased across different calorie ranges.

### 9. Beverage Category Proportions
We will visualize the proportion of each beverage category in our dataset using a pie chart.

In [ ]:
category_counts = df['Beverage_category'].value_counts()

# Create a pie chart with percentage labels and an attractive color palette
plt.figure(figsize=(10, 8))
plt.pie(
    category_counts,
    labels=category_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('pastel', len(category_counts)),
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
plt.title('Proportion of Beverage Categories in Starbucks Menu', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 10. Predicting Calories for 10 Sample Drinks
We will take a random sample of 10 drinks from our test partition (`X_test`), align them with their original names from `df`, and compare the actual calorie values to the predicted values from our trained Decision Tree model.

In [ ]:
# Select a sample of 10 indices from the test set
sample_indices = X_test.sample(10, random_state=42).index

# Extract names and original features
sample_drinks = df.loc[sample_indices, ['Beverage_category', 'Beverage', 'Beverage_prep']]
sample_features = X_test.loc[sample_indices]

# Make predictions
sample_predictions = dt_model.predict(sample_features)

# Construct comparison DataFrame
comparison_df = sample_drinks.copy()
comparison_df['Actual Calories'] = y_test.loc[sample_indices]
comparison_df['Predicted Calories'] = sample_predictions.round(1)
comparison_df['Difference (Error)'] = (comparison_df['Actual Calories'] - comparison_df['Predicted Calories']).round(1)

# Display the results
display(comparison_df)

### 11. Saving the Trained Model
We will serialize and save our trained `DecisionTreeRegressor` (`dt_model`) to a file using Python's `pickle` library.

In [ ]:
import pickle

# Define the file path where the model will be saved
model_filename = '/content/starbucks_calorie_predictor.pkl'

# Save the model to disk
with open(model_filename, 'wb') as file:
    pickle.dump(dt_model, file)

print(f"Success! The trained model has been saved to: {model_filename}")